# Full Model Evaluation

Evaluate end-to-end full-pipeline accuracy on all available data using `TennisFullDataset` and `InferencePipeline`.

In [11]:
from pathlib import Path
import sys
import torch
import pandas as pd
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

In [12]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists() and (candidate / "models").exists():
            return candidate
    raise RuntimeError("Could not find project root from current notebook location.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data.full_dataset import TennisFullDataset
from inference.inference import InferencePipeline
from models.bbox_detection import BBoxDetectionModel
from models.keypoint_detection import KeypointDetectionModel
from models.keypoint_attention_detection import KeypointAttentionDetectionModel
from models.pose_classification import PoseClassificationModel

EXPORT_DIR = PROJECT_ROOT / "exports"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "evaluation_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ROOT_CANDIDATES = [
    PROJECT_ROOT / "datasets" / "orvile" / "tennis-player-actions-dataset" / "versions" / "1" / "Tennis Player Actions Dataset for Human Pose Estimation",
    PROJECT_ROOT / "datasets" / "walnut",
]

DATASET_ROOT = next((path for path in DATASET_ROOT_CANDIDATES if path.exists()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError("Could not find dataset root in known locations.")

ANNOTATION_FILES = [
    "annotations/backhand.json",
    "annotations/forehand.json",
    "annotations/ready_position.json",
    "annotations/serve.json",
]

BBOX_CKPT_CANDIDATES = [EXPORT_DIR / "bbox_best.pt", CHECKPOINT_DIR / "bbox_best.pt"]
KEYPOINT_CKPT_CANDIDATES = [
    EXPORT_DIR / "keypoint_attention_best_state_dict.pt",
    CHECKPOINT_DIR / "keypoint_attention_best.pt",
    EXPORT_DIR / "keypoint_best_state_dict.pt",
    CHECKPOINT_DIR / "keypoint_best.pt",
]
# Prioritize predicted-keypoint Stage-3 model, then fallback to original pose model
POSE_CKPT_CANDIDATES = [
    EXPORT_DIR / "pose_best_predicted.pt",
    CHECKPOINT_DIR / "pose_best_predicted.pt",
    EXPORT_DIR / "pose_best.pt",
    CHECKPOINT_DIR / "pose_best.pt",
]

def pick_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

BBOX_CKPT = pick_existing(BBOX_CKPT_CANDIDATES)
KEYPOINT_CKPT = pick_existing(KEYPOINT_CKPT_CANDIDATES)
POSE_CKPT = pick_existing(POSE_CKPT_CANDIDATES)

if BBOX_CKPT is None or KEYPOINT_CKPT is None or POSE_CKPT is None:
    raise FileNotFoundError("Missing one or more required checkpoints (bbox/keypoint/pose).")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Device: {device}")
print(f"BBox checkpoint: {BBOX_CKPT}")
print(f"Keypoint checkpoint: {KEYPOINT_CKPT}")
print(f"Pose checkpoint: {POSE_CKPT}")

Project root: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection
Dataset root: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/datasets/orvile/tennis-player-actions-dataset/versions/1/Tennis Player Actions Dataset for Human Pose Estimation
Device: mps
BBox checkpoint: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/exports/bbox_best.pt
Keypoint checkpoint: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/exports/keypoint_attention_best_state_dict.pt
Pose checkpoint: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/exports/pose_best_predicted.pt


In [13]:
bbox_model = BBoxDetectionModel().to(device)
bbox_state = torch.load(BBOX_CKPT, map_location=device)
if not isinstance(bbox_state, dict):
    raise ValueError("Unexpected bbox checkpoint format. Expected state_dict dict.")
bbox_model.load_state_dict(bbox_state)
bbox_model.eval()

keypoint_payload = torch.load(KEYPOINT_CKPT, map_location="cpu")
if isinstance(keypoint_payload, dict) and "model_state" in keypoint_payload:
    keypoint_state = keypoint_payload["model_state"]
    num_keypoints = int(keypoint_payload.get("num_keypoints", 18))
    keypoint_h = int(keypoint_payload.get("image_height", 128))
    keypoint_w = int(keypoint_payload.get("image_width", 128))
elif isinstance(keypoint_payload, dict):
    keypoint_state = keypoint_payload
    num_keypoints = 18
    keypoint_h, keypoint_w = 128, 128
else:
    raise ValueError("Unexpected keypoint checkpoint format.")

keypoint_ckpt_name = KEYPOINT_CKPT.name.lower()
is_attention_keypoint = ("attention" in keypoint_ckpt_name) or (
    isinstance(keypoint_payload, dict) and "attention" in str(keypoint_payload.get("model_type", "")).lower()
)

if is_attention_keypoint:
    keypoint_model = KeypointAttentionDetectionModel(num_keypoints=num_keypoints).to(device)
else:
    keypoint_model = KeypointDetectionModel(num_keypoints=num_keypoints).to(device)
keypoint_model.load_state_dict(keypoint_state)
keypoint_model.eval()

pose_checkpoint = torch.load(POSE_CKPT, map_location="cpu")
if not isinstance(pose_checkpoint, dict) or "model_state" not in pose_checkpoint:
    raise ValueError("Unexpected pose checkpoint format. Expected checkpoint dict with model_state.")

pose_state = pose_checkpoint["model_state"]
pose_label_names = pose_checkpoint.get("label_names", None)
if pose_label_names is None:
    pose_label_names = ["backhand", "forehand", "ready_position", "serve"]

# Defaults aligned with latest pose_best_predicted training run
DEFAULT_POSE_ARGS = {
    "hidden_dim": 384,
    "dropout": 0.4,
    "visibility_threshold": 0.0,
}
pose_args = pose_checkpoint.get("args", {}) if isinstance(pose_checkpoint.get("args", {}), dict) else {}

pose_model = PoseClassificationModel(
    num_keypoints=num_keypoints,
    num_classes=len(pose_label_names),
    hidden_dim=int(pose_args.get("hidden_dim", DEFAULT_POSE_ARGS["hidden_dim"])),
    dropout=float(pose_args.get("dropout", DEFAULT_POSE_ARGS["dropout"])),
    visibility_threshold=float(pose_args.get("visibility_threshold", DEFAULT_POSE_ARGS["visibility_threshold"])),
).to(device)
pose_model.load_state_dict(pose_state)
pose_model.eval()

keypoint_image_size = (keypoint_h, keypoint_w)
pipeline = InferencePipeline(
    bbox_detection=bbox_model,
    keypoint_detection=keypoint_model,
    pose_detection=pose_model,
    bbox_image_size=(256, 256),
    keypoint_image_size=keypoint_image_size,
)

print("Models loaded and pipeline initialized.")
print("Keypoint model:", keypoint_model.__class__.__name__)
print("Pose label names:", pose_label_names)
print("Keypoint image size (H, W):", keypoint_image_size)
print("Pose architecture:", {
    "hidden_dim": int(pose_args.get("hidden_dim", DEFAULT_POSE_ARGS["hidden_dim"])),
    "dropout": float(pose_args.get("dropout", DEFAULT_POSE_ARGS["dropout"])),
    "visibility_threshold": float(pose_args.get("visibility_threshold", DEFAULT_POSE_ARGS["visibility_threshold"])),
})

Models loaded and pipeline initialized.
Keypoint model: KeypointAttentionDetectionModel
Pose label names: ['backhand', 'forehand', 'ready_position', 'serve']
Keypoint image size (H, W): (128, 128)
Pose architecture: {'hidden_dim': 384, 'dropout': 0.4, 'visibility_threshold': 0.0}


In [14]:
def _norm_label(name: str) -> str:
    return str(name).strip().lower().replace(" ", "_")

# Build canonical category names from annotation files
annotation_category_names = []
seen_names = set()
for rel_path in ANNOTATION_FILES:
    ann_path = DATASET_ROOT / rel_path
    with ann_path.open("r", encoding="utf-8") as f:
        data = __import__("json").load(f)
    for category in data.get("categories", []):
        name = category.get("name")
        if isinstance(name, str) and name not in seen_names:
            seen_names.add(name)
            annotation_category_names.append(name)

canonical_by_norm = {_norm_label(name): name for name in annotation_category_names}
class_order_for_dataset = []
missing_pose_labels = []
for label in pose_label_names:
    canonical = canonical_by_norm.get(_norm_label(label))
    if canonical is None:
        missing_pose_labels.append(label)
    else:
        class_order_for_dataset.append(canonical)

if missing_pose_labels:
    raise ValueError(
        f"Could not map pose labels to annotation categories: {missing_pose_labels}. "
        f"Available categories: {annotation_category_names}"
    )

eval_dataset = TennisFullDataset(
    root_dir=str(DATASET_ROOT),
    annotation_files=ANNOTATION_FILES,
    transform=transforms.ToTensor(),
    class_order=class_order_for_dataset,
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
)

print(f"Total evaluation samples: {len(eval_dataset)}")
print("Pose label names (model order):", pose_label_names)
print("Dataset class order (canonical):", eval_dataset.label_names)

Total evaluation samples: 2000
Pose label names (model order): ['backhand', 'forehand', 'ready_position', 'serve']
Dataset class order (canonical): ['Backhand', 'Forehand', 'Ready_Position', 'Serve']


In [15]:
to_pil = transforms.ToPILImage()
correct = 0
total = 0
rows = []

pipeline.bbox_detection.eval()
pipeline.keypoint_detection.eval()
pipeline.pose_detection.eval()

with torch.no_grad():
    for sample_index, (image_tensor, label_tensor) in enumerate(tqdm(eval_loader, desc="Evaluating")):
        pil_image = to_pil(image_tensor[0].cpu())
        true_label = int(label_tensor.item())

        pose_probs = pipeline.run_pipe(pil_image)
        if pose_probs.dim() == 1:
            pose_probs = pose_probs.unsqueeze(0)

        pred_label = int(torch.argmax(pose_probs, dim=1).item())
        confidence = float(torch.max(pose_probs).item())

        total += 1
        if pred_label == true_label:
            correct += 1

        rows.append({
            "sample_index": sample_index,
            "image_path": eval_dataset.samples[sample_index]["img_path"],
            "true_label": true_label,
            "true_class": eval_dataset.label_names[true_label],
            "pred_label": pred_label,
            "pred_class": eval_dataset.label_names[pred_label],
            "confidence": confidence,
            "correct": int(pred_label == true_label),
        })

accuracy = correct / total if total > 0 else 0.0
results_df = pd.DataFrame(rows)

print(f"Correct: {correct}")
print(f"Total: {total}")
print(f"Accuracy: {accuracy:.4f}")

Evaluating: 100%|██████████| 2000/2000 [01:39<00:00, 20.03it/s]

Correct: 1058
Total: 2000
Accuracy: 0.5290


In [16]:
per_class_accuracy = (
    results_df.groupby("true_class")["correct"]
    .mean()
    .sort_index()
    .rename("accuracy")
)

display(per_class_accuracy.to_frame())

,accuracy
true_class,
Backhand,0.656
Forehand,0.496
Ready_Position,0.578
Serve,0.386


In [17]:
predictions_path = OUTPUT_DIR / "test_predictions.csv"
summary_path = OUTPUT_DIR / "full_model_accuracy_summary.csv"

results_df.to_csv(predictions_path, index=False)
summary_df = pd.DataFrame([
    {
        "samples": total,
        "correct": correct,
        "accuracy": accuracy,
    }
])
summary_df.to_csv(summary_path, index=False)

print(f"Saved predictions: {predictions_path}")
print(f"Saved summary: {summary_path}")
results_df.head()

Saved predictions: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/evaluation_outputs/test_predictions.csv
Saved summary: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/evaluation_outputs/full_model_accuracy_summary.csv


,sample_index,image_path,true_label,true_class,pred_label,pred_class,confidence,correct
0,0,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,0,Backhand,0,Backhand,0.989332,1
1,1,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,0,Backhand,0,Backhand,0.825151,1
2,2,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,0,Backhand,0,Backhand,0.679177,1
3,3,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,0,Backhand,0,Backhand,0.593601,1
4,4,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,0,Backhand,0,Backhand,0.582612,1
